# PI-DeepONet / SWE manuscript revision — combined run (Kaggle)

One notebook that runs, end to end and unattended, the whole revision programme:

| Part | Source | Blocker addressed |
|---|---|---|
| **1** | `01_reference_solver_audit.ipynb` | Blocker 1 — reference solver, CFL, convergence, shock, data regeneration |
| **2** | `02_attractor_theory.ipynb` | Blocker 2 — the corrected Proposition 1 and its verification |
| **3** | `03_metrics_ablation_speedup.ipynb` | Blocker 3 — metrics, fusion ablation, honest speedup |
| **4** | `pi_deeponet_swe_v6.ipynb` | the manuscript's own 40k pipeline re-run on the new data, the BC × IC × residual factorial, and the gradient norm re-measured in v6's code |

Part 1 writes `swe_data_wb.npz`, which Parts 3 and 4 consume. Part 2 is independent of
both. Parts 2–3 use `deeponet_tf.py` (a minimal reimplementation built for
diagnostics); Part 4 uses `pi_deeponet_v6.py`, a faithful port of the paper's own
architecture and training loop, so its numbers drop straight into the manuscript.

---

## Kaggle setup — three steps

**1. Make the repo visible to the notebook.** The notebook needs `swe_solvers.py`,
`deeponet_tf.py` and `pi_deeponet_v6.py` (all in the repo's `new/` folder). Any one of
these works:

* **Dataset (most reliable, no internet needed).** Zip the repo (or just the `new/`
  folder), upload it via *Datasets → New Dataset*, then in this notebook use
  *Add Input → your dataset*. It lands under `/kaggle/input/<slug>/…` and the bootstrap
  cell finds it automatically, at any nesting depth.
* **Git clone (already wired up).** `REPO_URL` in the config cell is set to this
  repo, so you only need *Settings → Internet: On* — the bootstrap cell clones it
  into `/kaggle/working/repo` for you. This is the step that most often bites:
  if Internet is off, the clone cannot run and the bootstrap stops immediately.
* **Manual.** `!git clone …` in a cell of your own before running the bootstrap, or
  point the `SCIML_REPO` environment variable at the checkout.

**2. Turn on the GPU.** *Settings → Accelerator → GPU T4 x2* (only one is used).
Parts 2 and 3 train networks; Part 1 is pure NumPy and CPU-bound either way.

**3. Smoke-test, then commit.** Set `QUICK = True` in the config cell and *Run All*
once (~4 minutes) to prove the whole chain works in your environment. Then set it back
to `False` and use *Save Version → Save & Run All (Commit)* for the real run.

> On a committed run, a cell that raises stops everything after it. That is why the
> `QUICK` pass matters — it exercises every cell at small sizes first.

## Expected wall-clock for the full run (`QUICK = False`)

| Section | Cost |
|---|---|
| 1.4 converged reference, `nx = 12800` | **10–25 min** (single-threaded NumPy — the most expensive non-training cell) |
| 1.3 convergence + shock diagnostics | ~5 min |
| 1.6 data regeneration (152 trajectories) | ~1 min |
| 2.3 PI training, 3 cases × 3000 steps | ~10 min on a T4 |
| 3.3 fusion ablation, 3 × 15000 steps | ~30–45 min on a T4 |
| 3.5 speedup benchmark | ~5 min |
| 4.1 the 40k production run, ×2 IC modes | ~20–60 min on a T4 |
| 4.2–4.4 error table, factorial, gradient norms | ~10 min |

Roughly **2–2.5 hours** total on a T4, comfortably inside Kaggle's 12-hour limit.
`CFG["IC_MODES_40K"] = ("paper",)` halves §4.1 if you only want the published shortcut.

## What gets written

Everything lands in `/kaggle/working` (downloadable from the *Output* tab):

```
swe_data_wb.npz        regenerated training data + honest generation cost
run_log.txt            full console transcript of the run
results.json           machine-readable summary of every headline number
figures/*.png          every figure, at 150 dpi
models/*.weights.h5    the three fusion variants and the 40k production models
```

## 0. Configuration

`QUICK = True` shrinks every expensive knob so the whole notebook runs in a few minutes.
Use it to validate the environment, then set it back to `False`.

In [ ]:
import os, sys, json, time, shutil, subprocess, importlib.util
from pathlib import Path

# ----------------------------------------------------------------- knobs
QUICK = False          # True -> ~4 min smoke test; False -> the real run

REPO_URL = "https://github.com/phoenixfin/sciml-framework.git"
                       # cloned only if the support modules are not already
                       # visible; needs Kaggle "Internet" switched on. Set it
                       # to "" if you attach the repo as a Dataset instead.

RUN_PART1 = True       # reference-solver audit + data regeneration
RUN_PART2 = True       # attractor theory (needs TensorFlow)
RUN_PART3 = True       # metrics / ablation / speedup (needs Part 1's npz + TensorFlow)
RUN_PART4 = True       # the manuscript's own v6 pipeline, re-run on the new data

CFG = dict(
    SEED            = 42,
    NX_REF          = 12800,                      # 1.4 converged reference grid
    CONV_NXS        = (200, 400, 800, 1600, 3200),# 1.3 self-convergence ladder
    CONV_TIMES      = (0.25, 0.5, 1.0),
    SHOCK_NXS       = (400, 800, 1600, 3200, 6400),
    NX_DATA         = 400,                        # 1.6 data grid
    N_TRAIN         = 502,                        # GP samples drawn
    N_SUP           = 152,                        # supervised trajectories solved
    M_SENSORS       = 100,                        # branch sensor count
    P_BASIS         = 64,                         # trunk/branch latent width
    N_COLLOC        = 4000,                       # 2.x PDE collocation points
    PI_STEPS        = 3000,                       # 2.3 physics-only training steps
    FUSION_STEPS    = 15000,                      # 3.3 supervised training steps
    SPEEDUP_BATCHES = (1, 10, 100),               # 3.5 benchmark batch sizes
    ITER_40K        = 40000,                      # 4.1 the manuscript's training budget
    IC_MODES_40K    = ("paper", "exp"),           # 4.1 IC shortcut variants to train
    N_TEST_C4       = 100,                        # 4.2 unseen operator-generalisation pairs
    PI_FACTORIAL_STEPS = 3000,                    # 4.3 steps per factorial cell
)

if QUICK:
    CFG.update(
        NX_REF=1600, CONV_NXS=(200, 400, 800), CONV_TIMES=(0.25, 1.0),
        SHOCK_NXS=(400, 800, 1600), N_TRAIN=64, N_SUP=32,
        N_COLLOC=1000, PI_STEPS=200, FUSION_STEPS=400, SPEEDUP_BATCHES=(1, 10),
        ITER_40K=600, IC_MODES_40K=("paper",), N_TEST_C4=8, PI_FACTORIAL_STEPS=100,
    )

# ----------------------------------------------------- output directory
ON_KAGGLE = Path("/kaggle/working").is_dir()
OUT_DIR = Path("/kaggle/working") if ON_KAGGLE else Path.cwd() / "revision_outputs"
FIG_DIR, MODEL_DIR = OUT_DIR / "figures", OUT_DIR / "models"
for p in (OUT_DIR, FIG_DIR, MODEL_DIR):
    p.mkdir(parents=True, exist_ok=True)
DATA_NPZ = OUT_DIR / "swe_data_wb.npz"
RESULTS = {"quick_mode": QUICK, "config": {k: list(v) if isinstance(v, tuple) else v
                                           for k, v in CFG.items()}}

# ------------------------------------------- mirror the console to a file
class _Tee:
    """Write to the notebook stream and to run_log.txt at the same time."""
    def __init__(self, stream, fh):
        self._s, self._f = stream, fh
    def write(self, s):
        n = self._s.write(s)
        try:
            self._f.write(s); self._f.flush()
        except Exception:
            pass
        return n
    def flush(self):
        self._s.flush()
        try:
            self._f.flush()
        except Exception:
            pass
    def __getattr__(self, k):
        return getattr(self._s, k)

if not globals().get("_TEE_INSTALLED"):
    try:
        _LOG_FH = open(OUT_DIR / "run_log.txt", "a", encoding="utf-8", buffering=1)
        sys.stdout, sys.stderr = _Tee(sys.stdout, _LOG_FH), _Tee(sys.stderr, _LOG_FH)
        _TEE_INSTALLED = True
    except Exception as e:                                    # never fatal
        print("run_log.txt not available:", e)

# ------------------------------------- prefer Keras 2 (requirements.txt pins <2.17)
if importlib.util.find_spec("tf_keras") is not None:
    os.environ.setdefault("TF_USE_LEGACY_KERAS", "1")
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "2")

print(f"QUICK mode : {QUICK}")
print(f"on Kaggle  : {ON_KAGGLE}")
print(f"output dir : {OUT_DIR}")
print(f"parts      : 1={RUN_PART1}  2={RUN_PART2}  3={RUN_PART3}  4={RUN_PART4}")

## 0.1 Locate the repository

Finds `swe_solvers.py` and `deeponet_tf.py` wherever you attached them — the working
directory and its parents, anywhere under `/kaggle/input`, the `SCIML_REPO` environment
variable, or a fresh clone of `REPO_URL`. Also puts the repo's `src/` on `sys.path` so
`import sciml` works if you want the packaged code.

In [ ]:
SUPPORT_FILES = ("swe_solvers.py", "deeponet_tf.py", "pi_deeponet_v6.py")


def _holds_support(p: Path) -> bool:
    try:
        return all((p / f).is_file() for f in SUPPORT_FILES)
    except OSError:
        return False


def _walk_dirs(root: Path, max_depth: int = 5):
    """Directories under `root`, breadth-limited, skipping noise."""
    root = Path(root)
    if not root.is_dir():
        return
    skip = {".git", "__pycache__", ".ipynb_checkpoints", "node_modules",
            ".pytest_cache", ".ruff_cache", "site-packages"}
    frontier = [(root, 0)]
    while frontier:
        d, depth = frontier.pop(0)
        yield d
        if depth >= max_depth:
            continue
        try:
            kids = [c for c in d.iterdir() if c.is_dir() and c.name not in skip]
        except OSError:
            continue
        frontier.extend((c, depth + 1) for c in kids)


def find_support_dir(extra_roots=()):
    """Cheap exact checks first, then a bounded walk of every plausible root."""
    here = Path.cwd().resolve()
    seeds = [here, *here.parents]
    env = os.environ.get("SCIML_REPO")
    if env:
        seeds.insert(0, Path(env))
    for s in seeds:
        for cand in (s, s / "new"):
            if _holds_support(cand):
                return cand.resolve()
    roots = [Path(r) for r in extra_roots] + [
        Path("/kaggle/input"), Path("/kaggle/working"), Path("/kaggle/usr/lib"), here]
    for root in roots:
        for cand in _walk_dirs(root):
            if _holds_support(cand):
                return cand.resolve()
    return None


SUPPORT_DIR = find_support_dir()
clone_note = "not attempted (modules already present)" if SUPPORT_DIR else "not attempted"

if SUPPORT_DIR is None and REPO_URL:
    dest = Path("/kaggle/working/repo") if ON_KAGGLE else Path.cwd() / "repo"
    if dest.exists():
        clone_note = f"{dest} already exists, reused"
    else:
        print(f"cloning {REPO_URL} -> {dest} ...")
        r = subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(dest)],
                           capture_output=True, text=True)
        clone_note = ("ok" if r.returncode == 0 else
                      f"FAILED (exit {r.returncode}): {(r.stderr or '').strip()[-500:]}")
        print("git clone:", clone_note)
    SUPPORT_DIR = find_support_dir([dest])          # search the clone itself

if SUPPORT_DIR is None:
    seen = []
    for root in (Path("/kaggle/input"), Path("/kaggle/working")):
        if root.is_dir():
            seen += [f"    {p}" for p in sorted(root.iterdir())[:20]]
    raise RuntimeError(
        f"Could not find {', '.join(SUPPORT_FILES)}.\n"
        f"  git clone : {clone_note}\n"
        f"  REPO_URL  : {REPO_URL or '(empty)'}\n"
        f"  cwd       : {Path.cwd()}\n"
        "Fix it with any one of:\n"
        "  * Settings -> Internet: On, so REPO_URL can be cloned (most common cause:\n"
        "    Internet is off, or the repo is private and needs a Dataset instead)\n"
        "  * Add Input -> a Kaggle dataset holding the repo (or just its new/ folder)\n"
        "  * os.environ['SCIML_REPO'] = '/path/to/checkout' before this cell\n"
        + ("visible input/working entries:\n" + "\n".join(seen) if seen else ""))

sys.path.insert(0, str(SUPPORT_DIR))

# repo root = the directory holding src/, if we can see it
REPO_ROOT = next((p for p in (SUPPORT_DIR, *SUPPORT_DIR.parents)
                  if (p / "src" / "sciml").is_dir()), None)
if REPO_ROOT is not None:
    sys.path.insert(0, str(REPO_ROOT / "src"))

print(f"support modules : {SUPPORT_DIR}")
print(f"repo root       : {REPO_ROOT}")
RESULTS["support_dir"] = str(SUPPORT_DIR)

## 0.2 Shared imports and helpers

Everything below this cell is the merged content of notebooks 01–03, in order. Import
statements that the three notebooks duplicated live here instead.

In [ ]:
import numpy as np
import matplotlib
import matplotlib.pyplot as plt

from swe_solvers import (lxf_paper, lxf_viscosity, swe_solve, cell_centers,
                         rel_l2, rel_l2_anomaly, G)

plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 150, "font.size": 9})
np.set_printoptions(precision=4, suppress=True)

L, T = 10.0, 1.0
h0_C1 = lambda x: 1.0 + 0.5 * np.exp(-2.0 * (x - 5.0) ** 2)
b_C2  = lambda x: 0.2 * np.exp(-(x - 5.0) ** 2)

_T_START = time.time()


def banner(title):
    print("\n" + "=" * 78)
    print(f"  {title}    [t+{time.time() - _T_START:7.1f}s]")
    print("=" * 78)


def show_fig(name):
    """Save the current figure into OUT_DIR/figures and display it."""
    fig = plt.gcf()
    path = FIG_DIR / f"{name}.png"
    fig.savefig(path, bbox_inches="tight")
    plt.show()
    print(f"[fig] {path}")


def error_triplet(pred, ref, h_rest=None):
    """The three error metrics Part 3 recommends reporting side by side."""
    r = ref.mean() if h_rest is None else h_rest
    return dict(rel_total=rel_l2(pred, ref),
                rel_anomaly=rel_l2_anomaly(pred, ref, r),
                rmse_m=float(np.sqrt(np.mean((pred - ref) ** 2))))


def periodic_se(xs_, sigma, ell, Lp=L, jitter=1e-8):
    """Periodic squared-exponential kernel, as in §3.6."""
    d_ = np.abs(xs_[:, None] - xs_[None, :])
    return sigma ** 2 * np.exp(-2.0 * np.sin(np.pi * d_ / Lp) ** 2 / ell ** 2) \
           + jitter * np.eye(xs_.size)


def sample_gp(xs_, n, sigma, ell, mean, rng, clip_lo=None):
    """Draw n periodic GP samples on xs_ (used for training and for the C4 test set)."""
    Kc = np.linalg.cholesky(periodic_se(xs_, sigma, ell))
    s = mean + (Kc @ rng.standard_normal((xs_.size, n))).T
    return s if clip_lo is None else np.maximum(s, clip_lo)


def save_results():
    with open(OUT_DIR / "results.json", "w", encoding="utf-8") as fh:
        json.dump(RESULTS, fh, indent=2, default=float)


print("numpy", np.__version__, "| matplotlib", matplotlib.__version__)
print(f"L = {L} m, T = {T} s, g = {G} m/s^2")

---

# Part 1 — Reference solver audit and data regeneration

**Addresses Blocker 1 (reference solver) and supplies the numerics C&F will ask for.**

| # | Experiment | Replaces / adds |
|---|---|---|
| 1 | CFL audit of the Lax-Friedrichs configuration | corrects §3.3 ("CFL ≈ 0.45") |
| 2 | Well-balanced HLL solver + lake-at-rest test | **new** — Table W1, §4 |
| 3 | Grid-convergence / self-convergence study | **new** — Table W2, replaces the O(Δx²/Δt) claim |
| 4 | Shock-formation diagnostic for benchmark C1 | **new** — see "Shock" below |
| 5 | LxF vs converged reference error budget | corrects §3.3 error-budget paragraph |
| 6 | Data regeneration + honest generation cost | updates §3.6, Table 1, abstract |
| 7 | Conservation diagnostics (mass / momentum) | **new** — Fig. W1 |

Needs only `numpy`, `matplotlib` and `swe_solvers.py`.

## 1.1 CFL audit of the manuscript's Lax-Friedrichs configuration

§3.3 states that `nx = 400`, `nt = 4000` gives "a CFL number of approximately 0.45".
Measure it, and compute the resulting modified-equation viscosity

$$\nu_{\mathrm{LxF}} = \frac{\Delta x^2}{2\Delta t}\,(1-\mathrm{CFL}^2)$$

Lax-Friedrichs is the one scheme whose numerical diffusion *grows* as $\Delta t$ is
reduced at fixed $\Delta x$, so an under-estimated CFL is not a harmless bookkeeping error.

In [ ]:
if RUN_PART1:
    banner("1.1  CFL audit of the manuscript's Lax-Friedrichs configuration")

    nx = 400
    x400 = cell_centers(L, nx)
    q_lxf, cfl_meas, dx, dt = lxf_paper(x400, h0_C1(x400), np.zeros(nx), T, nt=4000)
    nu = lxf_viscosity(dx, dt, cfl_meas)

    print(f"dx = {dx:.4e} m,  dt = {dt:.4e} s")
    print(f"measured max CFL           : {cfl_meas:.4f}   (manuscript claims 0.45)")
    print(f"numerical viscosity nu_LxF : {nu:.4f} m^2/s")
    print(f"diffusion length sqrt(4*nu*T) = {np.sqrt(4 * nu * T):.3f} m"
          f"   (Gaussian half-width ~ 0.7 m, domain L = {L} m)")

    # what nt SHOULD be for the stated CFL
    c_typ = np.sqrt(G * 1.0) + 0.5
    nt_target = int(np.ceil(T / (0.45 * dx / c_typ)))
    print(f"\nnt required for CFL = 0.45 : {nt_target}  (manuscript used 4000)")

    RESULTS["cfl_audit"] = dict(dx=dx, dt=dt, cfl_measured=cfl_meas,
                                nu_lxf=nu, nt_for_cfl_045=nt_target)

## 1.2 Well-balanced HLL reference solver

`swe_solve` implements Audusse-type hydrostatic reconstruction with an HLL flux,
`order=1` (Euler) or `order=2` (minmod-MUSCL on $(\eta, hu, b)$ with SSP-RK2).
Reconstructing the free surface $\eta = h+b$ rather than $h$ is what makes it
well-balanced.

**Lake-at-rest test** — $h_0 - b = \text{const}$, $u = 0$ must be preserved exactly.
This is the first test an SWE referee runs, and the manuscript currently has no
equivalent. It is also rhetorically useful: it is the one configuration where the
trivial $F=0$ state is the *correct* answer.

In [ ]:
if RUN_PART1:
    banner("1.2  Table W1 - lake at rest")

    rows = []
    for order in (1, 2):
        for nx_ in (200, 400):
            xx = cell_centers(L, nx_)
            bb = b_C2(xx)
            q, _ = swe_solve(xx, 1.5 - bb, bb, T, cfl=0.45, order=order)
            rows.append((order, nx_,
                         np.max(np.abs(q[0] + bb - 1.5)),
                         np.max(np.abs(q[1]))))

    print("Table W1 - lake at rest (h0 - b = 1.5, u = 0), t = 1 s")
    print(f"{'order':>6}{'nx':>7}{'max|eta-1.5| [m]':>20}{'max|hu| [m^2/s]':>18}")
    for o, n, e, m in rows:
        print(f"{o:>6}{n:>7}{e:>20.3e}{m:>18.3e}")
    print("\n-> well balanced to machine precision")

    RESULTS["lake_at_rest"] = [dict(order=o, nx=n, max_eta_err=e, max_hu=m)
                               for o, n, e, m in rows]

## 1.3 Grid convergence

Two studies. **Self-convergence** (successive refinement) certifies the solver
order without needing a truth solution; **absolute convergence** against a fine run
gives the number you quote as the reference-data error floor.

Run the pre-shock time first (`T = 0.25`, `T = 0.5`) — that is where a formal
order can legitimately be claimed.

In [ ]:
if RUN_PART1:
    banner("1.3  Table W2 - self-convergence")

    def self_convergence(Tend, nxs=CFG["CONV_NXS"], order=2):
        prev, out = None, []
        for nx_ in nxs:
            xx = cell_centers(L, nx_)
            q, _ = swe_solve(xx, h0_C1(xx), np.zeros(nx_), Tend, cfl=0.45, order=order)
            if prev is not None:
                xc, hc = prev
                out.append((nx_ // 2, rel_l2(hc, np.interp(xc, xx, q[0], period=L))))
            prev = (xx, q[0].copy())
        return out

    conv_all = {}
    print("Table W2 - self-convergence of the order-2 well-balanced HLL solver")
    for Tend in CFG["CONV_TIMES"]:
        res = self_convergence(Tend)
        conv_all[Tend] = res
        print(f"\n  T = {Tend} s")
        print(f"  {'nx':>7}{'rel L2':>13}{'order':>9}")
        for i, (n, e) in enumerate(res):
            r = "" if i == 0 else f"{np.log2(res[i - 1][1] / e):.2f}"
            print(f"  {n:>7}{e:>13.3e}{r:>9}")

    RESULTS["self_convergence"] = {str(k): [dict(nx=n, rel_l2=e) for n, e in v]
                                   for k, v in conv_all.items()}

### The order collapses at $T = 1$ s — and that is a physical result, not a bug

At $T=0.25$ and $T=0.5$ the scheme is clean second order. At $T=1$ it drops to
$\approx 0.5$. The reason is below: **benchmark C1 is not smooth at $t = 1$ s.**
A shock forms at around $t \approx 0.75$–$0.8$ s.

If the maximum gradient *doubles* every time the grid is halved, the solution has a
genuine discontinuity; if it saturates, the feature is merely steep.

In [ ]:
if RUN_PART1:
    banner("1.3b  Shock-formation diagnostic for benchmark C1")

    print("max|dh/dx| at T = 1 s vs resolution")
    grad_vs_nx = []
    for nx_ in CFG["SHOCK_NXS"]:
        xx = cell_centers(L, nx_)
        q, _ = swe_solve(xx, h0_C1(xx), np.zeros(nx_), T, cfl=0.45, order=2)
        gmax = np.max(np.abs(np.diff(q[0], append=q[0][0]))) / (L / nx_)
        grad_vs_nx.append((nx_, float(gmax)))
        print(f"  nx = {nx_:5d}   max|dh/dx| = {gmax:8.2f}")

    nx_ = max(CFG["SHOCK_NXS"][:-1])
    xx = cell_centers(L, nx_)
    snaps = [round(0.1 * i, 2) for i in range(1, 11)]
    _, out = swe_solve(xx, h0_C1(xx), np.zeros(nx_), T, cfl=0.45, order=2, snapshots=snaps)
    print(f"\nsteepening history (nx = {nx_})")
    hist_steep = []
    for t_ in snaps:
        h, _ = out[t_]
        gmax = np.max(np.abs(np.diff(h, append=h[0]))) / (L / nx_)
        hist_steep.append((t_, float(gmax), float(h.max() - h.min())))
        print(f"  t = {t_:.1f}   max|dh/dx| = {gmax:8.2f}"
              f"   peak-to-peak h = {h.max() - h.min():.4f}")

    RESULTS["shock"] = dict(grad_vs_nx=grad_vs_nx, steepening=hist_steep)

**Consequence for the manuscript.** The paper describes C1 as a smooth benchmark
and attributes the oscillations in Fig. 2 at $t=1.0$ s to "the finite spectral
resolution of the tanh trunk MLP". The real cause is that the operator is being asked
to represent a **discontinuity** with a smooth architecture — Gibbs ringing around a
genuine shock. The huge LxF viscosity was smearing the shock and hiding this.

Two honest options, both defensible:
1. Keep $T = 1$ s, state that C1 develops a shock at $t \approx 0.78$ s, and reframe the
   $t=1$ oscillations as shock-related. This *strengthens* the paper — you now have a
   shock-capturing result rather than a smooth-only one.
2. Shorten the horizon to $T = 0.6$ s so C1 genuinely stays smooth, and move the
   shock case to a separate labelled benchmark.

Option 1 is more interesting for a C&F audience.

## 1.4 Error budget: LxF reference vs converged reference

Replaces the §3.3 claim that the LxF truncation error is $\approx 5\times10^{-4}$ m
and therefore negligible against the operator error.

> **This is the slowest cell in the notebook.** `nx = 12800` on a single NumPy thread
> takes 10–25 minutes; `QUICK` drops it to 1600.

In [ ]:
if RUN_PART1:
    banner(f"1.4  Converged reference at nx = {CFG['NX_REF']}  (slow)")

    nxr = CFG["NX_REF"]
    xr = cell_centers(L, nxr)
    t0 = time.time()
    qr, _ = swe_solve(xr, h0_C1(xr), np.zeros(nxr), T, cfl=0.45, order=2)
    ref_seconds = time.time() - t0
    print(f"converged reference (nx = {nxr}) computed in {ref_seconds:.0f} s")
    ref400 = np.interp(x400, xr, qr[0], period=L)

    # same 400-cell grid, but at the CFL the paper says it used
    nt45 = nt_target
    q45, cfl45, _, dt45 = lxf_paper(x400, h0_C1(x400), np.zeros(nx), T, nt=nt45)
    q_hll, _ = swe_solve(x400, h0_C1(x400), np.zeros(nx), T, cfl=0.45, order=2)

    budget = []
    print(f"\n{'scheme':<34}{'CFL':>8}{'nu [m2/s]':>12}{'relL2(h)':>12}"
          f"{'relL2(anom)':>14}{'p2p h':>9}")
    for name, q, c_, nu_ in [
            ("LxF nx=400 nt=4000 (manuscript)", q_lxf, cfl_meas, nu),
            (f"LxF nx=400 nt={nt45} (CFL 0.45)", q45, cfl45, lxf_viscosity(dx, dt45, cfl45)),
            ("well-balanced HLL o2, nx=400",     q_hll, 0.45, 0.0)]:
        e_tot, e_anom = rel_l2(q[0], ref400), rel_l2_anomaly(q[0], ref400)
        budget.append(dict(scheme=name, cfl=c_, nu=nu_, rel_l2=e_tot,
                           rel_l2_anomaly=e_anom, p2p=float(q[0].max() - q[0].min())))
        print(f"{name:<34}{c_:>8.3f}{nu_:>12.4f}{e_tot:>12.3e}"
              f"{e_anom:>14.3e}{q[0].max() - q[0].min():>9.4f}")
    print(f"{'converged reference':<34}{'-':>8}{'-':>12}{'-':>12}{'-':>14}"
          f"{qr[0].max() - qr[0].min():>9.4f}")

    RESULTS["error_budget"] = dict(nx_ref=nxr, ref_seconds=ref_seconds, rows=budget,
                                   ref_p2p=float(qr[0].max() - qr[0].min()))

In [ ]:
if RUN_PART1:
    fig, ax = plt.subplots(1, 2, figsize=(12, 4))
    ax[0].plot(xr, qr[0], 'k-', lw=2, label=f'converged ref (nx={nxr}, o2)')
    ax[0].plot(x400, q_lxf[0], 'r--', label='LxF nx=400 nt=4000 (manuscript)')
    ax[0].plot(x400, q45[0], 'b-.', label=f'LxF nx=400 nt={nt45} (CFL 0.45)')
    ax[0].plot(x400, q_hll[0], 'g:', lw=2, label='WB-HLL o2 nx=400')
    ax[0].set_xlabel('x [m]'); ax[0].set_ylabel('h [m]'); ax[0].set_title('h at T = 1 s')
    ax[0].legend(fontsize=8)

    ax[1].semilogy(x400, np.abs(q_lxf[0] - ref400) + 1e-12, 'r-', label='LxF (manuscript)')
    ax[1].semilogy(x400, np.abs(q45[0] - ref400) + 1e-12, 'b-', label='LxF CFL 0.45')
    ax[1].semilogy(x400, np.abs(q_hll[0] - ref400) + 1e-12, 'g-', label='WB-HLL o2')
    ax[1].set_xlabel('x [m]'); ax[1].set_ylabel('|h - h_ref| [m]')
    ax[1].set_title('pointwise reference error'); ax[1].legend(fontsize=8)
    plt.tight_layout()
    show_fig("p1_error_budget")

## 1.5 Mass and momentum conservation

A conservation-law paper in a CFD journal needs a conservation diagnostic. Run this
for the *reference* here; Part 3 runs the same diagnostic on the operator prediction,
which is where it actually bites.

In [ ]:
if RUN_PART1:
    banner("1.5  Fig. W1 - conservation diagnostics for the reference solver")

    nx_ = 800
    xx = cell_centers(L, nx_)
    dxx = L / nx_
    bb = b_C2(xx)
    snaps = [round(0.05 * i, 2) for i in range(1, 21)]
    _, out = swe_solve(xx, h0_C1(xx), bb, T, cfl=0.45, order=2, snapshots=snaps)
    M0 = np.sum(h0_C1(xx)) * dxx
    mass = [abs(np.sum(out[t_][0]) * dxx - M0) / M0 for t_ in snaps]
    mom  = [abs(np.sum(out[t_][1]) * dxx) for t_ in snaps]

    fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
    ax[0].semilogy(snaps, np.array(mass) + 1e-18, 'o-')
    ax[0].set_xlabel('t [s]'); ax[0].set_ylabel('|ΔM| / M₀')
    ax[0].set_title('relative mass drift')
    ax[1].semilogy(snaps, np.array(mom) + 1e-18, 's-', color='C1')
    ax[1].set_xlabel('t [s]'); ax[1].set_ylabel(r'$|\int hu\,dx|$')
    ax[1].set_title('total momentum (non-flat bed: not conserved)')
    plt.tight_layout()
    show_fig("p1_conservation_reference")
    print(f"final relative mass drift: {mass[-1]:.3e}")

    RESULTS["reference_conservation"] = dict(t=snaps, rel_mass_drift=mass,
                                             total_momentum=mom)

## 1.6 Regenerate the training set

Same periodic-squared-exponential GP sampler as §3.6, but with the well-balanced
solver at CFL 0.45. Report the honest generation cost — it should now be **lower**
than the 66 s quoted in the paper, because CFL 0.45 needs ~300 steps rather than
4000. That strengthens the data-efficiency argument rather than weakening it.

In [ ]:
if RUN_PART1:
    banner("1.6  Regenerate the training set")

    rng = np.random.default_rng(CFG["SEED"])
    NX_DATA, N_TRAIN, N_SUP = CFG["NX_DATA"], CFG["N_TRAIN"], CFG["N_SUP"]
    xg = cell_centers(L, NX_DATA)
    H0 = sample_gp(xg, N_TRAIN, 0.4, 2.0, 1.0, rng, clip_lo=0.3)
    BB = sample_gp(xg, N_TRAIN, 0.12, 3.0, 0.0, rng, clip_lo=0.0)
    H0[0], BB[0] = h0_C1(xg), np.zeros(NX_DATA)          # C1
    H0[1], BB[1] = h0_C1(xg), b_C2(xg)                   # C2
    H0 = np.maximum(H0, BB + 0.05 + 1e-3)                # enforce h0 > b + hmin

    print(f"max boundary gap  h0: {np.max(np.abs(H0[:, 0] - H0[:, -1])):.2e} m,"
          f"  b: {np.max(np.abs(BB[:, 0] - BB[:, -1])):.2e} m")

    snap_t = [0.25, 0.50, 0.75, 1.00]
    t0 = time.time()
    _, snaps_out = swe_solve(xg, H0[:N_SUP], BB[:N_SUP], T, cfl=0.45,
                             order=2, snapshots=snap_t)
    gen_time = time.time() - t0
    H_snap  = np.stack([snaps_out[t_][0] for t_ in snap_t], axis=1)   # (N_SUP, 4, nx)
    HU_snap = np.stack([snaps_out[t_][1] for t_ in snap_t], axis=1)
    print(f"\n{N_SUP} supervised trajectories (ensemble-vectorised): {gen_time:.1f} s "
          f"= {1000 * gen_time / N_SUP:.0f} ms each")
    print("snapshot tensor shapes:", H_snap.shape, HU_snap.shape)

    np.savez_compressed(DATA_NPZ, x=xg, h0=H0, b=BB,
                        t_snap=np.array(snap_t), h=H_snap, hu=HU_snap,
                        n_sup=N_SUP, gen_seconds=gen_time)
    print("saved ->", DATA_NPZ)

    RESULTS["data_generation"] = dict(n_train=N_TRAIN, n_sup=N_SUP, nx=NX_DATA,
                                      seconds_total=gen_time,
                                      ms_per_trajectory=1000 * gen_time / N_SUP,
                                      path=str(DATA_NPZ))
    save_results()

## 1.7 Error-metric utilities

Blocker 3: $\varepsilon_h$ normalised by $\|h\|_2$ is flattered by the ~1 m constant
background. Quantify the inflation factor so you can decide what to report.

In [ ]:
if RUN_PART1:
    banner("1.7  Metric inflation from the constant background depth")

    h_ref_field = qr[0]
    infl = {}
    for label, field in [("converged reference", h_ref_field),
                         ("manuscript LxF field", q_lxf[0])]:
        inf_rest = np.linalg.norm(field) / np.linalg.norm(field - 1.0)
        inf_mean = np.linalg.norm(field) / np.linalg.norm(field - field.mean())
        infl[label] = dict(vs_rest=float(inf_rest), vs_mean=float(inf_mean))
        print(f"{label:<24} ||h||/||h-h_rest|| = {inf_rest:5.1f}x,"
              f"  ||h||/||h-mean|| = {inf_mean:5.1f}x")

    eq = 1.17e-2 * np.linalg.norm(q_lxf[0]) / np.linalg.norm(q_lxf[0] - q_lxf[0].mean())
    print(f"\nSo the manuscript's eps_h = 1.17e-2 corresponds to roughly {eq:.2f}"
          " on the free-surface anomaly.")
    print("Recommended reporting: relative L2 on eta = h - h_rest, PLUS dimensional RMSE in metres.")

    RESULTS["metric_inflation"] = dict(factors=infl, eps_h_1p17e_2_as_anomaly=float(eq))
    save_results()

### Checklist of manuscript edits Part 1 supports

- §3.3 — replace the CFL number, replace the $O(\Delta x^2/\Delta t)$ truncation-error
  sentence with Table W2, replace LxF with the well-balanced HLL scheme throughout.
- §3.3 / §5 — delete "the dominant error source is the operator approximation rather
  than the reference solver diffusion"; the measured numbers invert this.
- §3.6, Table 1, abstract — update the data-generation cost.
- §4 — add Table W1 (lake at rest), Table W2 (convergence), Fig. W1 (conservation).
- §4.2 — reframe the $t=1$ s oscillations around shock formation; delete "capturing over
  98.8% of the spatial variance".
- Table 2 — annotate C1/C2 as "smooth until $t \approx 0.78$ s, shock thereafter".

---

# Part 2 — The attractor: corrected statement and its verification

**Addresses Blocker 2.** Proposition 1 as written claims $\nabla_\theta L_{PDE} = 0$ at
$F=0$. The branch-parameter half of the argument does not hold:

$$\nabla_{\theta_{\text{branch}}} L_{PDE}
= 2R\,\frac{\partial R}{\partial F}\,\underbrace{\frac{\partial F}{\partial \boldsymbol\beta}}_{=\;\boldsymbol\tau\;\neq\;0}\,
\frac{\partial \boldsymbol\beta}{\partial\theta}$$

There is no factor of $\boldsymbol\beta$ in this chain, so nothing forces it to vanish.

### What is actually true (the claim to prove instead)

1. $\nabla_{\theta_{\text{trunk}}} L_{PDE} = 0$ **exactly** — the trunks are frozen.
2. The **mass residual vanishes identically**, $R_1 \equiv 0$, and so does its gradient.
   Half the physics loss is structurally blind. *This is the hyperbolic-specific part*
   and it is what fails to hold for diffusion–reaction.
3. The surviving gradient comes only from $R_2$, and it points toward
   $\partial_x(\tfrac12 g h^2) = -gh\,\partial_x b$ — the **lake-at-rest steady-state
   manifold**, not the wave dynamics.
4. In the well-balanced case $h_0 - b = \text{const}$, $F = 0$ is an **exact global
   minimum** of $L_{PDE}$.

Experiments 1–3 verify each clause. Experiment 5 fixes the IC shortcut.

> Prior art to cite: Rohrhofer, Posch, Gößnitzer & Geiger, *On the Role of Fixed Points of
> Dynamical Systems in Training PINNs*, TMLR 2023 (arXiv:2203.13648); De Ryck, Mishra &
> Molinaro, *wPINNs*, on hyperbolic conservation laws.

In [ ]:
if RUN_PART2 or RUN_PART3 or RUN_PART4:
    banner("2.0  TensorFlow environment")

    import tensorflow as tf
    from deeponet_tf import FourBranchDeepONet, swe_residual, gradient_split_at_F0

    print("TensorFlow", tf.__version__)
    print("keras      ", getattr(tf.keras, "__version__", "?"),
          "(legacy)" if os.environ.get("TF_USE_LEGACY_KERAS") == "1" else "(keras 3)")
    gpus = tf.config.list_physical_devices("GPU")
    print("GPUs       ", [g.name for g in gpus] or "none — this will be slow")
    for g in gpus:                                  # avoid grabbing all VRAM up front
        try:
            tf.config.experimental.set_memory_growth(g, True)
        except Exception:
            pass

    tf.keras.utils.set_random_seed(CFG["SEED"])
    M = CFG["M_SENSORS"]
    xs = np.linspace(0.0, L, M, endpoint=False).astype(np.float32)   # sensor points
    xs_t = tf.constant(xs[None, :])
    RESULTS["tensorflow"] = dict(version=tf.__version__, gpus=[g.name for g in gpus],
                                 legacy_keras=os.environ.get("TF_USE_LEGACY_KERAS") == "1")

## 2.1 Trunk / branch gradient split at $F = 0$

`gradient_split_at_F0` zeroes every branch output layer (forcing $\boldsymbol\beta = 0$,
hence $F \equiv 0$) and reports the two gradient norms separately.

**Predictions:** trunk norm exactly 0 in every case; mass residual exactly 0 in every
case; branch norm nonzero except for lake at rest.

This single table replaces Proposition 1's proof and is worth putting in the paper.

In [ ]:
if RUN_PART2:
    banner("2.1  Trunk / branch gradient split at F = 0")

    def make_case(name, h0_fn, b_fn):
        return dict(name=name, h0=h0_fn, b=b_fn)

    CASES = [
        make_case("C1  flat bed, h0 = 1+0.5 exp(-2(x-5)^2)",
                  lambda x: 1.0 + 0.5 * tf.exp(-2.0 * (x - 5.0) ** 2),
                  lambda x: tf.zeros_like(x)),
        make_case("C2  bump bathymetry",
                  lambda x: 1.0 + 0.5 * tf.exp(-2.0 * (x - 5.0) ** 2),
                  lambda x: 0.2 * tf.exp(-(x - 5.0) ** 2)),
        make_case("LAKE AT REST  h0 - b = 1.5 (exact steady state)",
                  lambda x: 1.5 - 0.2 * tf.exp(-(x - 5.0) ** 2),
                  lambda x: 0.2 * tf.exp(-(x - 5.0) ** 2)),
        make_case("FLAT WATER  h0 = 1, b = 0 (also a steady state)",
                  lambda x: tf.ones_like(x),
                  lambda x: tf.zeros_like(x)),
    ]

    NC = CFG["N_COLLOC"]
    rng2 = np.random.default_rng(0)
    xc = tf.constant(rng2.uniform(0, L, (NC, 1)).astype(np.float32))
    tc = tf.constant(rng2.uniform(0, T, (NC, 1)).astype(np.float32))

    model = FourBranchDeepONet(m=M, p=CFG["P_BASIS"], fusion="add", ic_mode="paper")
    _ = model([tf.tile(xs_t, [NC, 1]) * 0 + 1.0, tf.tile(xs_t, [NC, 1]) * 0,
               tf.ones((NC, 1)), tf.zeros((NC, 1)), xc, tc])   # build

    split_rows = []
    print(f"{'case':<48}{'||g_trunk||':>13}{'||g_branch||':>14}{'rms R1':>11}{'rms R2':>11}")
    for c in CASES:
        h0s = tf.tile(c['h0'](xs_t), [NC, 1])
        bs  = tf.tile(c['b'](xs_t),  [NC, 1])
        r = gradient_split_at_F0(model, h0s, bs, c['h0'], c['b'], xc, tc)
        split_rows.append(dict(case=c['name'], **r))
        print(f"{c['name']:<48}{r['trunk_grad_norm']:>13.3e}{r['branch_grad_norm']:>14.3e}"
              f"{r['mass_residual_rms']:>11.3e}{r['momentum_residual_rms']:>11.3e}")

    RESULTS["gradient_split_at_F0"] = split_rows
    save_results()

**How to read the table.** If `||g_trunk||` is 0 everywhere, `rms R1` is 0 everywhere,
`||g_branch||` is nonzero for C1/C2 and ~0 for the two steady states, then the corrected
statement holds and the original Proposition 1 does not. Report exactly this table.

## 2.2 The surviving gradient points at hydrostatic balance

Decompose $L_{PDE}$ into its mass and momentum halves and confirm that at $F=0$ the
momentum residual equals the hydrostatic imbalance
$\partial_x(\tfrac12 g h_0^2) + g h_0 \partial_x b$ — i.e. the loss is measuring
*departure from lake at rest*, not departure from the true wave solution.

In [ ]:
if RUN_PART2:
    banner("2.2  The surviving gradient points at hydrostatic balance")

    def hydrostatic_imbalance(h0_fn, b_fn, x):
        with tf.GradientTape(persistent=True) as g:
            g.watch(x)
            h0, b = h0_fn(x), b_fn(x)
            press = 0.5 * G * h0 ** 2
        def _d(y, v):
            gr = g.gradient(y, v)
            return tf.zeros_like(v) if gr is None else gr
        r = _d(press, x) + G * h0 * _d(b, x)
        del g
        return r

    hyd_rows = []
    for c in CASES:
        h0s = tf.tile(c['h0'](xs_t), [NC, 1])
        bs  = tf.tile(c['b'](xs_t),  [NC, 1])
        # force F = 0 by zeroing branch output layers
        saved = []
        for out_ in ("h", "hu"):
            for net in model.branch[out_]:
                w, bv = net.layers[-1].get_weights()
                saved.append((net, w.copy(), bv.copy()))
                net.layers[-1].set_weights([np.zeros_like(w), np.zeros_like(bv)])
        r1, r2 = swe_residual(model, h0s, bs, c['h0'], c['b'], xc, tc)
        for net, w, bv in saved:
            net.layers[-1].set_weights([w, bv])
        hyd = hydrostatic_imbalance(c['h0'], c['b'], xc)
        rel = float(tf.norm(r2 - hyd) / (tf.norm(r2) + 1e-12))
        hyd_rows.append(dict(case=c['name'], rel_diff=rel))
        print(f"{c['name'][:44]:<46} ||R2 - hydrostatic imbalance|| / ||R2|| = {rel:.3e}")

    RESULTS["hydrostatic_match"] = hyd_rows

## 2.3 PI-DeepONet training: balanced vs unbalanced

The decisive dynamical test. Train the *purely physics-informed* variant on:

- **(a)** a lake-at-rest configuration → $F=0$ is an exact minimum, collapse should be total;
- **(b)** the C1/C2 configurations → gradient is nonzero, so the model should move, but
  toward hydrostatic balance rather than the wave solution.

If (b) shows the model drifting to a *steady state that is not $h_0$* — rather than sitting
exactly at $h_0$ as the paper reports — the reformulated claim is confirmed and the
"exact stationary point" language must go.

Models are cached in `PI_MODELS` so the next cell can reuse the C2 run rather than
retraining it.

In [ ]:
if RUN_PART2:
    banner(f"2.3  Physics-only training, {CFG['PI_STEPS']} steps per case")

    def pde_loss(model_, h0s, bs, h0_fn, b_fn, x, t):
        r1, r2 = swe_residual(model_, h0s, bs, h0_fn, b_fn, x, t)
        return tf.reduce_mean(r1 ** 2) + tf.reduce_mean(r2 ** 2)

    def train_pi(case, steps=None, lr=1e-3, log_every=None):
        steps = CFG["PI_STEPS"] if steps is None else steps
        log_every = max(1, steps // 12) if log_every is None else log_every
        tf.keras.utils.set_random_seed(0)
        mdl = FourBranchDeepONet(m=M, p=CFG["P_BASIS"], fusion="add", ic_mode="exp")
        opt = tf.keras.optimizers.Adam(lr)
        h0s = tf.tile(case['h0'](xs_t), [NC, 1])
        bs  = tf.tile(case['b'](xs_t),  [NC, 1])
        hist = []
        for k in range(steps + 1):
            with tf.GradientTape() as tape:
                Lp = pde_loss(mdl, h0s, bs, case['h0'], case['b'], xc, tc)
            gs = tape.gradient(Lp, mdl.trainable_variables)
            gs, gn = tf.clip_by_global_norm(gs, 1.0)
            opt.apply_gradients(zip(gs, mdl.trainable_variables))
            if k % log_every == 0 or k == steps:
                beta_n = float(tf.norm(mdl.beta("h", h0s[:32], bs[:32]), axis=-1).numpy().mean())
                hist.append((k, float(Lp), float(gn), beta_n))
        return mdl, hist

    PI_MODELS, PI_HIST = {}, {}
    for c in (CASES[2], CASES[0], CASES[1]):     # lake at rest first
        t0 = time.time()
        mdl, hist = train_pi(c)
        PI_MODELS[c['name']], PI_HIST[c['name']] = mdl, hist
        print(f"\n{c['name']}   ({time.time() - t0:.0f}s)")
        print(f"  {'step':>6}{'L_PDE':>13}{'||grad||':>12}{'mean ||beta_h||':>18}")
        for k, lp, gn, bn in hist:
            print(f"  {k:>6}{lp:>13.4e}{gn:>12.4e}{bn:>18.4e}")

    RESULTS["pi_training"] = {name: [dict(step=k, L_pde=lp, grad=gn, beta=bn)
                                     for k, lp, gn, bn in h]
                              for name, h in PI_HIST.items()}
    save_results()

In [ ]:
if RUN_PART2:
    banner("2.3b  Where did the unbalanced run actually end up?")

    xq = tf.constant(np.linspace(0, L, 500, dtype=np.float32)[:, None])
    tq = tf.ones_like(xq) * T
    c = CASES[1]
    mdl = PI_MODELS[c['name']]                      # reuse, do not retrain
    h0s = tf.tile(c['h0'](xs_t), [500, 1]); bs = tf.tile(c['b'](xs_t), [500, 1])
    h_pred, hu_pred = mdl([h0s, bs, c['h0'](xq), c['b'](xq), xq, tq])
    h_pred = h_pred.numpy().ravel()

    xn = xq.numpy().ravel()
    h0n = c['h0'](xq).numpy().ravel(); bn_ = c['b'](xq).numpy().ravel()
    lake = (h0n + bn_).mean() - bn_          # the lake-at-rest state with the same volume

    plt.figure(figsize=(7, 4))
    plt.plot(xn, h0n, 'k--', label=r'$h_0(x)$  (the "$F=0$" state)')
    plt.plot(xn, lake, 'g-.', label='lake at rest, same volume')
    plt.plot(xn, h_pred, 'r-', lw=2, label='PI-DeepONet at T=1 s')
    plt.xlabel('x [m]'); plt.ylabel('h [m]'); plt.legend()
    plt.title('Where does PI training go?')
    plt.tight_layout()
    show_fig("p2_attractor_endpoint")

    d_h0, d_lake = float(np.linalg.norm(h_pred - h0n)), float(np.linalg.norm(h_pred - lake))
    print(f"||h_pred - h0||   = {d_h0:.4f}")
    print(f"||h_pred - lake|| = {d_lake:.4f}")
    print("-> if the second is smaller, the attractor is the steady-state manifold, not h0.")

    RESULTS["attractor_endpoint"] = dict(dist_to_h0=d_h0, dist_to_lake=d_lake,
                                         lake_is_closer=d_lake < d_h0)

## 2.4 Re-measure the PDE gradient norm properly

§3.7.3 reports $\|\nabla_\theta L_{PDE}\| \approx 6.6\times10^{12}$, Remark 3 reports
$1.5\times10^2$ by finite differences, and the Fig. 6 caption says $2.2\times10^1$. Three
numbers, three places. A referee will read $10^{12}$ as a bug (division by a near-zero
$h$, or a norm over an unreduced loss).

Measure it per-network, at the paper's initialisation, with the ELU floor active.

In [ ]:
if RUN_PART2:
    banner("2.4  One consistently measured PDE gradient norm")

    tf.keras.utils.set_random_seed(CFG["SEED"])
    mdl = FourBranchDeepONet(m=M, p=CFG["P_BASIS"], fusion="add", ic_mode="paper")
    c = CASES[0]
    h0s = tf.tile(c['h0'](xs_t), [NC, 1]); bs = tf.tile(c['b'](xs_t), [NC, 1])

    with tf.GradientTape(persistent=True) as tape:
        Lp = pde_loss(mdl, h0s, bs, c['h0'], c['b'], xc, tc)
    groups = {"branch_h": [v for n in mdl.branch['h'] for v in n.trainable_variables],
              "branch_hu": [v for n in mdl.branch['hu'] for v in n.trainable_variables],
              "trunk_h": mdl.trunk['h'].trainable_variables,
              "trunk_hu": mdl.trunk['hu'].trainable_variables}
    print(f"L_PDE at init = {float(Lp):.4e}")
    gnorms = {}
    for name, vs in groups.items():
        g = [x for x in tape.gradient(Lp, vs) if x is not None]
        gnorms[name] = float(tf.linalg.global_norm(g))
        print(f"  ||grad|| {name:<11} = {gnorms[name]:.4e}")
    gnorms["TOTAL"] = float(tf.linalg.global_norm(
        [x for x in tape.gradient(Lp, mdl.trainable_variables) if x is not None]))
    print(f"  ||grad|| TOTAL      = {gnorms['TOTAL']:.4e}")
    del tape
    print("\nReport ONE number, from automatic differentiation, and make Fig. 6's caption match.")

    RESULTS["pde_gradient_norm"] = dict(L_pde_at_init=float(Lp), **gnorms)
    save_results()

## 2.5 Fix the IC shortcut

Two defects in Eq. (12):

- **Not exact at $t=0$**: $\hat h(x,0) = h_0(x) + \epsilon$, because $\epsilon = 10^{-4}$ is
  added outside the ELU.
- **Positivity is not guaranteed**: $\mathrm{elu}(z) > -1$, so
  $\hat h > b + h_{\min} + \epsilon - 1$, which is **negative for $b < 0.95$ m**. The stated
  floor $\hat h \ge b + h_{\min} + \epsilon$ is false.

`ic_mode` offers three replacements. Verify all four numerically.

In [ ]:
if RUN_PART2:
    banner("2.5  IC shortcut variants")

    xq = tf.constant(np.linspace(0, L, 400, dtype=np.float32)[:, None])
    c = CASES[1]
    h0q, bq = c['h0'](xq), c['b'](xq)

    ic_rows = []
    print(f"{'ic_mode':<10}{'max|h(x,0)-h0|':>18}{'min h over t':>16}{'guaranteed floor':>20}")
    for mode in ("paper", "shifted", "exp", "softplus"):
        tf.keras.utils.set_random_seed(1)
        mdl = FourBranchDeepONet(m=M, p=CFG["P_BASIS"], fusion="add", ic_mode=mode)
        h0s = tf.tile(c['h0'](xs_t), [400, 1]); bs = tf.tile(c['b'](xs_t), [400, 1])
        h_ic, hu_ic = mdl([h0s, bs, h0q, bq, xq, tf.zeros_like(xq)])
        ic_err = float(tf.reduce_max(tf.abs(h_ic - h0q)))

        # adversarial stress test: drive the correction field strongly negative
        Fstress = tf.fill(tf.shape(xq), tf.constant(-50.0))
        h_stress = mdl.shortcut_h(h0q, bq, Fstress, tf.ones_like(xq))
        min_h = float(tf.reduce_min(h_stress))
        floor_ok = bool(min_h >= float(tf.reduce_min(bq)) + 0.05 - 1e-5)
        ic_rows.append(dict(ic_mode=mode, ic_error=ic_err, min_h_stressed=min_h,
                            floor_respected=floor_ok))
        print(f"{mode:<10}{ic_err:>18.3e}{min_h:>16.4f}{str(floor_ok):>20}")
    print("\n'exp' and 'softplus' are exact at t=0 AND keep a hard floor at b + h_min.")

    RESULTS["ic_shortcut"] = ic_rows
    save_results()

### Text changes Part 2 supports

- Replace Proposition 1 with the four-clause statement at the top; the proof is three
  lines and is *correct*.
- Rewrite Remark 2: the hyperbolic/parabolic distinction is the identically-vanishing
  **mass** residual under $hu(x,0)=0$, not the chain-rule argument (which is
  equation-agnostic).
- Rewrite Remark 3 around one consistently measured gradient norm; fix Fig. 6's caption.
- Rename "$F=0$ attractor" → "steady-state (lake-at-rest) attractor" in the title,
  abstract, keywords and §3.5.1.
- Add the Rohrhofer et al. (2023) and De Ryck et al. citations and soften
  "not previously characterised".
- Replace Eq. (12) with the `exp` or `softplus` form and drop the softplus-underflow
  paragraph (with the floor in place, $u = hu/h$ can never see a near-zero denominator).

---

# Part 3 — Metrics, branch-fusion ablation, and an honest speedup

**Addresses Blocker 3 plus the secondary items a C&F referee will raise.**

| Experiment | Manuscript target |
|---|---|
| 1. Anomaly-normalised error metrics | §4.2–4.5, Table 3, abstract |
| 2. Conservation diagnostics on the operator | **new** — §4, Fig. W2 |
| 3. Branch-fusion ablation (add / concat / bilinear) | §3.4.2, Table 4 |
| 4. Strong-bump case C2b | §4.3 — tests the fusion claim properly |
| 5. Like-for-like speedup benchmark | §4.9, Table 5 |

Reads `swe_data_wb.npz` written by Part 1.

In [ ]:
if RUN_PART3:
    banner("3.0  Load the regenerated dataset")

    if not DATA_NPZ.exists():
        raise FileNotFoundError(
            f"{DATA_NPZ} is missing — run Part 1 (RUN_PART1 = True) first, "
            "or copy an existing swe_data_wb.npz into the output directory.")
    d = np.load(DATA_NPZ)
    xg, H0, BB = d["x"], d["h0"], d["b"]
    t_snap, H_snap, HU_snap = d["t_snap"], d["h"], d["hu"]
    N_SUP = int(d["n_sup"])
    print("data:", H_snap.shape, "| generation cost", float(d["gen_seconds"]), "s")

## 3.1 Error metrics that are not flattered by the background depth

Report all three. The first is what the paper currently reports; the second is what a
hydraulics referee considers meaningful; the third is unambiguous.

In [ ]:
if RUN_PART3:
    banner("3.1  Metric inflation, per snapshot time")

    print(f"{'t [s]':>7}{'||h||/||h-hbar||':>20}")
    infl_by_t = {}
    for i, t_ in enumerate(t_snap):
        f = H_snap[:, i, :]
        infl = np.linalg.norm(f, axis=1) / np.linalg.norm(f - f.mean(axis=1, keepdims=True), axis=1)
        infl_by_t[float(t_)] = float(infl.mean())
        print(f"{t_:>7.2f}{infl.mean():>20.1f}")
    print("\nMultiply any reported rel_total by these to get the error on the wave signal.")

    # demo on a synthetic 'prediction' = reference + smooth perturbation
    ref = H_snap[0, -1, :]
    pred = ref + 0.01 * np.sin(6 * np.pi * xg / L)
    print("\nexample:", {k: f"{v:.3e}" for k, v in error_triplet(pred, ref, 1.0).items()})

    RESULTS["metric_inflation_by_time"] = infl_by_t

**Recommended reporting for Table 3.** Keep `rel_total` for continuity with the
literature, add `rel_anomaly` and `rmse_m` as the primary columns, and state the
normalisation explicitly in the caption. Also add $\bar\varepsilon_{hu} = 2.28\times10^{-1}$
to the abstract — omitting it while quoting the depth figure reads as selective.

## 3.2 Conservation baseline for the reference solver

The operator's own conservation diagnostic needs a trained model, so it runs in §3.4
after the ablation. This cell establishes the reference curve to plot against.

In [ ]:
if RUN_PART3:
    banner("3.2  Reference-solver conservation baseline")

    def operator_conservation(predict_fn, h0, b, x, times):
        """predict_fn(h0, b, x, t) -> (h, hu) on the grid x at scalar time t.

        Returns relative mass drift and total-momentum history.
        """
        dx_ = x[1] - x[0]
        M0_ = np.sum(h0) * dx_
        mass_, mom_ = [], []
        for t_ in times:
            h, hu = predict_fn(h0, b, x, t_)
            mass_.append(abs(np.sum(h) * dx_ - M0_) / M0_)
            mom_.append(np.sum(hu) * dx_)
        return np.array(mass_), np.array(mom_)

    # reference behaviour for comparison (flat bed => momentum is conserved too)
    times = np.linspace(0.05, T, 20)
    _, out = swe_solve(xg, H0[0], BB[0], T, cfl=0.45, order=2,
                       snapshots=[float(t_) for t_ in times])
    dx = xg[1] - xg[0]; M0 = np.sum(H0[0]) * dx
    ref_mass = np.array([abs(np.sum(out[float(t_)][0]) * dx - M0) / M0 for t_ in times])
    print("reference solver final relative mass drift:", f"{ref_mass[-1]:.3e}")

    RESULTS["operator_conservation"] = dict(reference_final_drift=float(ref_mass[-1]))

## 3.3 Branch-fusion ablation

§3.4.2 says the $h_0$–$b$ interaction "is instead mediated through the shared trunk".
The trunks take only $(x,t)$ and never see $h_0$ or $b$, so they cannot mediate anything.
With $\boldsymbol\beta = B_1(h_0) + B_2(b)$ the coefficient map is **additively separable**,
while the source term $-gh\,\partial_x b$ is bilinear.

C2 does not refute this because $b$ has amplitude 0.12 m — the coupling is weak.
Train three fusion variants and evaluate on both C2 and a strong-bump C2b.

In [ ]:
if RUN_PART3:
    banner("3.3a  Build the supervised training tensors")

    NXQ = H_snap.shape[-1]

    def grid_interp_np(f, x_src, x_q):
        return np.interp(x_q, x_src, f, period=L)

    def make_dataset():
        h0s = np.stack([grid_interp_np(H0[j], xg, xs) for j in range(N_SUP)]).astype(np.float32)
        bs  = np.stack([grid_interp_np(BB[j], xg, xs) for j in range(N_SUP)]).astype(np.float32)
        return h0s, bs

    H0S, BS = make_dataset()
    XQ = xg.astype(np.float32)
    TS = t_snap.astype(np.float32)
    print("branch inputs:", H0S.shape, BS.shape)

In [ ]:
if RUN_PART3:
    banner(f"3.3b  Fusion ablation - training 3 variants x {CFG['FUSION_STEPS']} steps")

    def train_variant(fusion, steps=None, lam_hu=5.0, lam_bc=5.0,
                      p=None, seed=0, log=None):
        steps = CFG["FUSION_STEPS"] if steps is None else steps
        p = CFG["P_BASIS"] if p is None else p
        log = max(1, steps // 6) if log is None else log
        tf.keras.utils.set_random_seed(seed)
        mdl = FourBranchDeepONet(m=M, p=p, fusion=fusion, ic_mode="exp")
        opt = tf.keras.optimizers.Adam(
            tf.keras.optimizers.schedules.ExponentialDecay(1e-3, 10000, 0.5, staircase=True))

        NS, NX = len(TS), NXQ
        xq = tf.constant(np.tile(XQ, NS)[:, None])
        tq = tf.constant(np.repeat(TS, NX)[:, None])
        h0q = tf.constant(np.tile(H0[:N_SUP], (1, NS)).astype(np.float32))   # (N,NS*NX)
        bq  = tf.constant(np.tile(BB[:N_SUP], (1, NS)).astype(np.float32))
        href = tf.constant(H_snap.reshape(N_SUP, -1).astype(np.float32))
        huref = tf.constant(HU_snap.reshape(N_SUP, -1).astype(np.float32))
        h0s_t, bs_t = tf.constant(H0S), tf.constant(BS)

        def fwd(idx, xx, tt, hq, bqq):
            n = tf.shape(idx)[0]; nq = tf.shape(xx)[0]
            h0s = tf.repeat(tf.gather(h0s_t, idx), nq, axis=0)
            bs  = tf.repeat(tf.gather(bs_t,  idx), nq, axis=0)
            xr = tf.tile(xx, [n, 1]); tr = tf.tile(tt, [n, 1])
            return mdl([h0s, bs, tf.reshape(hq, [-1, 1]), tf.reshape(bqq, [-1, 1]), xr, tr])

        @tf.function(reduce_retracing=True)
        def step(idx, tbc):
            with tf.GradientTape() as tape:
                hh, hhu = fwd(idx, xq, tq, tf.gather(h0q, idx), tf.gather(bq, idx))
                hh = tf.reshape(hh, [tf.shape(idx)[0], -1])
                hhu = tf.reshape(hhu, [tf.shape(idx)[0], -1])
                Ld = tf.reduce_mean((hh - tf.gather(href, idx)) ** 2) \
                     + lam_hu * tf.reduce_mean((hhu - tf.gather(huref, idx)) ** 2)
                # periodic BC at x=0 and x=L
                x0 = tf.zeros_like(tbc); xL = tf.fill(tf.shape(tbc), tf.constant(L, tf.float32))
                h0_0 = tf.gather(h0s_t, idx)[:, :1]; b_0 = tf.gather(bs_t, idx)[:, :1]
                hA, huA = fwd(idx, x0, tbc, tf.tile(h0_0, [1, tf.shape(tbc)[0]]),
                              tf.tile(b_0, [1, tf.shape(tbc)[0]]))
                hB, huB = fwd(idx, xL, tbc, tf.tile(h0_0, [1, tf.shape(tbc)[0]]),
                              tf.tile(b_0, [1, tf.shape(tbc)[0]]))
                Lb = tf.reduce_mean((hA - hB) ** 2) + tf.reduce_mean((huA - huB) ** 2)
                Ltot = Ld + lam_bc * Lb
            g = tape.gradient(Ltot, mdl.trainable_variables)
            g, _ = tf.clip_by_global_norm(g, 1.0)
            opt.apply_gradients(zip(g, mdl.trainable_variables))
            return Ld, Lb

        rng_ = np.random.default_rng(seed)
        nbatch = min(8, N_SUP)
        t0 = time.time()
        for k in range(steps + 1):
            idx = tf.constant(rng_.choice(N_SUP, nbatch, replace=False).astype(np.int32))
            tbc = tf.constant(rng_.uniform(0, T, (64, 1)).astype(np.float32))
            Ld, Lb = step(idx, tbc)
            if k % log == 0 or k == steps:
                print(f"  [{fusion}] step {k:6d}  L_data {float(Ld):.3e}  L_bc {float(Lb):.3e}"
                      f"  ({time.time() - t0:.0f}s)")
        return mdl

    def save_weights_any(mdl_, stem):
        """Keras 3 wants a .weights.h5 suffix; Keras 2 also accepts the TF format."""
        for path in (MODEL_DIR / f"{stem}.weights.h5", MODEL_DIR / f"{stem}_ckpt"):
            try:
                mdl_.save_weights(str(path))
                return str(path)
            except Exception as e:
                last = e
        print(f"  [{stem}] weight save skipped: {last}")
        return None

    MODELS = {}
    for f in ("add", "concat", "bilinear"):
        MODELS[f] = train_variant(f)
        p_ = save_weights_any(MODELS[f], f"fusion_{f}")
        if p_:
            print(f"  [{f}] weights -> {p_}")

## 3.4 Conservation diagnostics on the operator prediction

Neural operators typically violate mass conservation at the percent level; showing that
you measured it is worth more than a good number, and hiding it is not an option at C&F.
The `add` (paper) variant is the one plotted.

In [ ]:
if RUN_PART3:
    banner("3.4  Fig. W2 - operator conservation vs the reference solver")

    def make_predict_fn(mdl_):
        def predict(h0, b, x, t_):
            n = x.size
            h0s = np.interp(xs, x, h0, period=L).astype(np.float32)[None, :]
            bs  = np.interp(xs, x, b,  period=L).astype(np.float32)[None, :]
            h, hu = mdl_([tf.tile(tf.constant(h0s), [n, 1]),
                          tf.tile(tf.constant(bs), [n, 1]),
                          tf.constant(h0.astype(np.float32))[:, None],
                          tf.constant(b.astype(np.float32))[:, None],
                          tf.constant(x.astype(np.float32))[:, None],
                          tf.fill((n, 1), np.float32(t_))])
            return h.numpy().ravel(), hu.numpy().ravel()
        return predict

    op_mass, op_mom = operator_conservation(make_predict_fn(MODELS["add"]),
                                            H0[0], BB[0], xg, times)

    plt.figure(figsize=(6.5, 4))
    plt.semilogy(times, ref_mass + 1e-18, 'k-', label='WB-HLL reference')
    plt.semilogy(times, op_mass + 1e-18, 'r-', label='DeepONet (add fusion)')
    plt.xlabel('t [s]'); plt.ylabel('|ΔM|/M₀'); plt.legend()
    plt.title('mass conservation: operator vs reference')
    plt.tight_layout()
    show_fig("p3_operator_conservation")

    print(f"operator final relative mass drift : {op_mass[-1]:.3e}")
    print(f"reference final relative mass drift: {ref_mass[-1]:.3e}")
    print(f"operator |total momentum| at T      : {abs(op_mom[-1]):.3e}  (flat bed => should be ~0)")

    RESULTS["operator_conservation"].update(
        times=[float(t_) for t_ in times],
        operator_mass_drift=[float(v) for v in op_mass],
        operator_momentum=[float(v) for v in op_mom],
        operator_final_drift=float(op_mass[-1]))
    save_results()

## 3.5 Strong-bump case C2b — where additive fusion should break

C2 uses a 0.2 m bump on 1 m of water (weak coupling). C2b uses 0.5 m, so the source
term $-gh\,\partial_x b$ genuinely depends on the *product* of the two inputs. If
`concat`/`bilinear` beat `add` on C2b but not on C2, that is direct evidence for the
architectural point and a clean new row in Table 4.

In [ ]:
if RUN_PART3:
    banner("3.5  Table 4 - fusion ablation across C1 / C2 / C2b")

    def evaluate(mdl_, h0_prof, b_prof, t_eval=T, nx=400):
        xq_ = cell_centers(L, nx).astype(np.float32)
        h0_ = np.interp(xq_, xg, h0_prof, period=L).astype(np.float32)
        b_  = np.interp(xq_, xg, b_prof,  period=L).astype(np.float32)
        ref_, _ = swe_solve(xq_.astype(float), h0_.astype(float), b_.astype(float),
                            float(t_eval), cfl=0.45, order=2)
        h0s = np.interp(xs, xq_, h0_, period=L).astype(np.float32)[None, :]
        bs  = np.interp(xs, xq_, b_,  period=L).astype(np.float32)[None, :]
        n = nx
        hp, hup = mdl_([tf.tile(tf.constant(h0s), [n, 1]), tf.tile(tf.constant(bs), [n, 1]),
                        tf.constant(h0_)[:, None], tf.constant(b_)[:, None],
                        tf.constant(xq_)[:, None], tf.fill((n, 1), np.float32(t_eval))])
        hp = hp.numpy().ravel(); hup = hup.numpy().ravel()
        return (error_triplet(hp, ref_[0], h_rest=float(ref_[0].mean())),
                error_triplet(hup, ref_[1], h_rest=0.0))

    xq_ = cell_centers(L, 400)
    CASES_EVAL = {
        "C1  flat bed":         (1 + 0.5 * np.exp(-2 * (xq_ - 5) ** 2), np.zeros_like(xq_)),
        "C2  bump 0.2 m":       (1 + 0.5 * np.exp(-2 * (xq_ - 5) ** 2), 0.2 * np.exp(-(xq_ - 5) ** 2)),
        "C2b bump 0.5 m (NEW)": (1.4 + 0.5 * np.exp(-2 * (xq_ - 5) ** 2), 0.5 * np.exp(-(xq_ - 5) ** 2)),
    }
    ablation = []
    print(f"{'case':<24}{'fusion':<10}{'eps_h(tot)':>12}{'eps_h(anom)':>13}"
          f"{'RMSE_h[m]':>12}{'eps_hu(tot)':>13}")
    for cname, (h0p, bp) in CASES_EVAL.items():
        for f, mdl_ in MODELS.items():
            eh, ehu = evaluate(mdl_, np.interp(xg, xq_, h0p, period=L),
                               np.interp(xg, xq_, bp, period=L))
            ablation.append(dict(case=cname, fusion=f, h=eh, hu=ehu))
            print(f"{cname:<24}{f:<10}{eh['rel_total']:>12.3e}{eh['rel_anomaly']:>13.3e}"
                  f"{eh['rmse_m']:>12.3e}{ehu['rel_total']:>13.3e}")

    RESULTS["fusion_ablation"] = ablation
    save_results()

## 3.6 A like-for-like speedup benchmark

Table 5 currently times a Python-loop CPU solver at **13× more timesteps than it
needs** against a batched GPU network. Two confounds, both inflating the number.

Fix all three legs:
- **CPU serial** — reference solver, one trajectory at a time (what a practitioner does today).
- **CPU ensemble-vectorised** — same solver, batched over the ensemble (fair software baseline).
- **GPU operator** — batched forward pass, timed after warm-up, with `.numpy()` sync.

Report all three. An honest $10^2$–$10^3\times$ is a good result; an unfalsifiable
$10^4\times$ is a referee magnet.

In [ ]:
if RUN_PART3:
    banner("3.6  Table 5 - three-leg speedup benchmark")

    def time_solver_serial(n, nx=400, reps=1):
        x_ = cell_centers(L, nx)
        rng_ = np.random.default_rng(1)
        h0_ = 1 + 0.3 * np.sin(2 * np.pi * (x_[None, :] + rng_.uniform(0, L, (n, 1))) / L)
        t0 = time.perf_counter()
        for r in range(reps):
            for j in range(n):
                swe_solve(x_, h0_[j], np.zeros(nx), T, cfl=0.45, order=2)
        return (time.perf_counter() - t0) / (reps * n) * 1e3      # ms per trajectory

    def time_solver_batch(n, nx=400, reps=1):
        x_ = cell_centers(L, nx)
        rng_ = np.random.default_rng(1)
        h0_ = 1 + 0.3 * np.sin(2 * np.pi * (x_[None, :] + rng_.uniform(0, L, (n, 1))) / L)
        B_ = np.zeros((n, nx))
        t0 = time.perf_counter()
        for r in range(reps):
            swe_solve(x_, h0_, B_, T, cfl=0.45, order=2)
        return (time.perf_counter() - t0) / (reps * n) * 1e3

    def time_operator(mdl_, n, nx=400, reps=5):
        x_ = cell_centers(L, nx).astype(np.float32)
        h0s = tf.constant(np.tile(np.interp(xs, x_, 1 + 0.3 * np.sin(2 * np.pi * x_ / L),
                                            period=L).astype(np.float32), (n * nx, 1)))
        bs = tf.zeros_like(h0s)
        h0q = tf.constant(np.tile((1 + 0.3 * np.sin(2 * np.pi * x_ / L)).astype(np.float32), n)[:, None])
        bq = tf.zeros_like(h0q)
        xq2 = tf.constant(np.tile(x_, n)[:, None]); tq2 = tf.fill(tf.shape(xq2), np.float32(T))
        _ = mdl_([h0s, bs, h0q, bq, xq2, tq2])                     # warm-up / trace
        t0 = time.perf_counter()
        for r in range(reps):
            h, hu = mdl_([h0s, bs, h0q, bq, xq2, tq2]); h.numpy()  # force sync
        return (time.perf_counter() - t0) / (reps * n) * 1e3

    mdl = MODELS["add"]
    speed_rows = []
    print(f"{'batch':>7}{'solver serial':>16}{'solver batched':>16}{'operator':>12}"
          f"{'speedup vs serial':>19}{'speedup vs batched':>20}")
    for n in CFG["SPEEDUP_BATCHES"]:
        ts, tb, to = time_solver_serial(n), time_solver_batch(n), time_operator(mdl, n)
        speed_rows.append(dict(batch=n, ms_serial=ts, ms_batched=tb, ms_operator=to,
                               speedup_vs_serial=ts / to, speedup_vs_batched=tb / to))
        print(f"{n:>7}{ts:>16.2f}{tb:>16.2f}{to:>12.4f}{ts / to:>19.0f}{tb / to:>20.0f}")
    print("\nAll times in ms per trajectory. State the CPU and GPU model in the caption,")
    print("and note that the solver leg is single-threaded numpy.")

    RESULTS["speedup"] = speed_rows
    save_results()

### Text changes Part 3 supports

- Table 3 / abstract — add anomaly-normalised errors and dimensional RMSE; add
  $\bar\varepsilon_{hu}$ to the abstract; delete "capturing over 98.8% of the spatial variance".
- §3.4.2 — delete the claim that the trunk mediates the $h_0$–$b$ interaction; state that
  additive fusion is separable and report the C2b ablation.
- Table 4 — add fusion variants as rows; report the true 10k-step numbers for A3 rather
  than repeating the 40k values.
- Table 5 / §4.9 — replace with the three-leg benchmark; reconcile every number in the
  prose against the table (the current text disagrees with it in seven places).
- §4 — add the operator conservation figure.

---

# Part 4 — The manuscript's own pipeline, re-run on the new data

Parts 2 and 3 use `deeponet_tf.py`, a minimal reimplementation built for
diagnostics. Part 4 uses `pi_deeponet_v6.py`, a faithful port of
`pi_deeponet_swe_v6.ipynb` — the paper's actual architecture, loss and training
loop — so that the numbers below are directly substitutable into the manuscript.

| § | Experiment | Manuscript target |
|---|---|---|
| 4.1 | 40k-step production run on the well-balanced data | abstract, Table 1, Table 3 |
| 4.2 | C1 / C2 / C3(OOD) / C4(unseen pairs) error table | Table 3 |
| 4.3 | BC-on/off × IC-mode factorial | resolves the $h_0$-vs-lake question |
| 4.4 | PDE gradient norm re-measured in v6's own code | §3.7.3, Remark 3, Fig. 6 caption |

### One thing to read before §4.3 and §4.4

v6's PDE residual (`pde_residual_fd`) is

$$R_1 = \partial_t h + \partial_x(hu), \qquad R_2 = \partial_t(hu)$$

$R_2$ is **only the time derivative**. The momentum flux divergence
$\partial_x(hu^2/h + \tfrac12 g h^2)$ and the bed source $gh\,\partial_x b$ are
absent, so what v6 minimises is not the SWE momentum equation.

That matters, because the truncated residual has an exact global minimum at
$F = 0$: driving $R_2 \to 0$ forces $hu \equiv 0$ (since $hu = tF_{hu}$ vanishes at
$t=0$), and then $R_1 = \partial_t h \to 0$ forces $h \equiv h_0$. **Proposition 1 is
true of v6's loss** — but that loss is not the shallow-water system, which is why
Part 2, using the full residual, lands on the lake-at-rest manifold instead.

§4.3 tests this directly: it crosses the requested BC-on/off × IC-mode 2×2 with a
third factor, the residual form, because the 2×2 alone cannot separate the two
hypotheses — all four cells share whichever residual you pick.

In [ ]:
if RUN_PART4:
    banner("4.0  Rebuild the v6 pipeline on the well-balanced data")

    import pi_deeponet_v6 as v6

    if not DATA_NPZ.exists():
        raise FileNotFoundError(
            f"{DATA_NPZ} is missing — run Part 1 (RUN_PART1 = True) first.")
    d4 = np.load(DATA_NPZ)
    xg4, H04, B4 = d4["x"], d4["h0"], d4["b"]
    Hs4, HUs4 = d4["h"], d4["hu"]
    ts4 = [float(t_) for t_ in d4["t_snap"]]
    nsup4 = int(d4["n_sup"])

    bundle = v6.build_bundle(xg4, H04, B4, Hs4, HUs4, ts4, n_sup=nsup4)
    print(f"pool {bundle.n_pool} trajectories, {bundle.n_sup} supervised, "
          f"snapshots {bundle.t_snaps}")
    print(f"v6 layout: {v6.M} sensors on [0,{v6.L_np}], data grid {v6.GRID} points")
    print(f"reference data: well-balanced HLL (was Lax-Friedrichs nx=400 nt=4000)")

## 4.1 The 40k-step run

Same architecture, same loss, same optimiser schedule, same budget as §8 of v6 —
only the supervised targets change, from the over-diffused LxF snapshots to the
well-balanced ones.

Both IC shortcuts are trained: `paper` is Eq. (12) as published, `exp` is the
replacement §2.5 recommends. Reporting the pair answers the obvious referee
question of what the corrected shortcut costs in accuracy. Drop `"exp"` from
`CFG["IC_MODES_40K"]` to halve this section.

In [ ]:
if RUN_PART4:
    banner(f"4.1  {CFG['ITER_40K']}-step production run(s)")

    MODELS40, HIST40 = {}, {}
    for ic in CFG["IC_MODES_40K"]:
        print(f"\n-- ic_mode = {ic} --")
        tf.keras.utils.set_random_seed(CFG["SEED"])
        mdl_ = v6.PIDeepONetSWE(ic_mode=ic).build_once()
        npar = int(sum(np.prod(v.shape) for v in mdl_.trainable_variables))
        print(f"  parameters: {npar:,}")
        HIST40[ic] = v6.train_model(mdl_, bundle, n_iter=CFG["ITER_40K"],
                                    seed=CFG["SEED"])
        MODELS40[ic] = mdl_
        for path in (MODEL_DIR / f"v6_40k_{ic}.weights.h5", MODEL_DIR / f"v6_40k_{ic}_ckpt"):
            try:
                mdl_.save_weights(str(path)); print(f"  weights -> {path}"); break
            except Exception as e:
                last = e
        else:
            print(f"  weight save skipped: {last}")

    RESULTS["run40k"] = {ic: dict(n_iter=CFG["ITER_40K"], n_params=npar,
                                  history=HIST40[ic]) for ic in MODELS40}
    save_results()

In [ ]:
if RUN_PART4:
    fig, ax = plt.subplots(1, 3, figsize=(11, 3.2))
    for ic, h_ in HIST40.items():
        ax[0].semilogy(h_["iter"], h_["Ld"], lw=1.2, label=f"{ic}")
        ax[1].semilogy(h_["iter"], [max(v, 1e-14) for v in h_["Lb"]], lw=1.2, label=f"{ic}")
        ax[2].plot(h_["iter"], h_["gnorm"], lw=1.2, label=f"{ic}")
    ax[0].set(xlabel="iteration", ylabel="$L_{data}$", title="data loss")
    ax[1].set(xlabel="iteration", ylabel="$L_{BC}$", title="periodic BC loss")
    ax[2].set(xlabel="iteration", ylabel=r"$\|\nabla\|$ (pre-clip)", title="gradient norm")
    for a in ax:
        a.legend(fontsize=8)
    plt.tight_layout()
    show_fig("p4_training_history")

## 4.2 Table 3, recomputed against the well-balanced reference

Four cases, three metrics each. `rel_total` is what the manuscript reports;
`rel_anomaly` and `rmse_m` are what §3.1 argues should be the primary columns.

C3 is the partial dam break, which is non-periodic and outside the training
distribution — as in v6, the periodic solver is still used for its reference, so
read it as a stress test rather than a clean benchmark.

In [ ]:
if RUN_PART4:
    banner("4.2  Table 3 - errors against the well-balanced reference")

    NXE = 400
    xe = cell_centers(L, NXE)

    def ref_solution(h0_arr, b_arr, t_eval=T):
        q, _ = swe_solve(xe, h0_arr, b_arr, float(t_eval), cfl=0.45, order=2)
        return q[0], q[1]

    def eval_batch(mdl_, h0_rows, b_rows, t_eval=T):
        """h0_rows, b_rows: (B, NXE) on xe. Returns predicted (h, hu), each (B, NXE)."""
        h0_sen = np.stack([np.interp(v6.x_sensors_np, xe, r, period=L) for r in h0_rows])
        b_sen = np.stack([np.interp(v6.x_sensors_np, xe, r, period=L) for r in b_rows])
        return v6.predict_at(mdl_, h0_sen, b_sen, xe, float(t_eval), h0_rows, b_rows)

    C_SINGLE = {
        "C1  smooth IC, flat bed":  (v6.h0_c1(xe), v6.b_flat(xe)),
        "C2  smooth IC, bump bed":  (v6.h0_c1(xe), v6.b_bump(xe)),
        "C3  dam break (OOD)":      (v6.h0_c3(xe), v6.b_flat(xe)),
    }

    # C4: unseen GP pairs, same sampler as §1.6, different seed
    rng4 = np.random.default_rng(CFG["SEED"] + 1000)
    NT4 = CFG["N_TEST_C4"]
    H0_t = sample_gp(xe, NT4, 0.4, 2.0, 1.0, rng4, clip_lo=0.3)
    B_t = sample_gp(xe, NT4, 0.12, 3.0, 0.0, rng4, clip_lo=0.0)
    H0_t = np.maximum(H0_t, B_t + 0.05 + 1e-3)
    t0 = time.time()
    q_t, _ = swe_solve(xe, H0_t, B_t, T, cfl=0.45, order=2)     # ensemble-vectorised
    print(f"C4: {NT4} unseen reference trajectories in {time.time() - t0:.0f}s")

    table3 = []
    print(f"\n{'case':<26}{'ic':<7}{'eps_h(tot)':>12}{'eps_h(anom)':>13}"
          f"{'RMSE_h[m]':>11}{'eps_hu(tot)':>13}{'eps_hu(anom)':>14}")
    for ic, mdl_ in MODELS40.items():
        for cname, (h0a, ba) in C_SINGLE.items():
            hr, hur = ref_solution(h0a, ba)
            hp, hup = eval_batch(mdl_, h0a[None, :], ba[None, :])
            eh = error_triplet(hp[0], hr, h_rest=float(hr.mean()))
            ehu = error_triplet(hup[0], hur, h_rest=0.0)
            table3.append(dict(case=cname, ic_mode=ic, h=eh, hu=ehu))
            print(f"{cname:<26}{ic:<7}{eh['rel_total']:>12.3e}{eh['rel_anomaly']:>13.3e}"
                  f"{eh['rmse_m']:>11.3e}{ehu['rel_total']:>13.3e}{ehu['rel_anomaly']:>14.3e}")

        hp, hup = eval_batch(mdl_, H0_t, B_t)
        per = [error_triplet(hp[k], q_t[0][k], h_rest=float(q_t[0][k].mean()))
               for k in range(NT4)]
        peru = [error_triplet(hup[k], q_t[1][k], h_rest=0.0) for k in range(NT4)]
        agg = {k: float(np.mean([p[k] for p in per])) for k in per[0]}
        aggu = {k: float(np.mean([p[k] for p in peru])) for k in peru[0]}
        agg["rel_total_std"] = float(np.std([p["rel_total"] for p in per]))
        table3.append(dict(case=f"C4  {NT4} unseen pairs (mean)", ic_mode=ic,
                           h=agg, hu=aggu))
        print(f"{'C4  ' + str(NT4) + ' unseen (mean)':<26}{ic:<7}{agg['rel_total']:>12.3e}"
              f"{agg['rel_anomaly']:>13.3e}{agg['rmse_m']:>11.3e}"
              f"{aggu['rel_total']:>13.3e}{aggu['rel_anomaly']:>14.3e}")

    print("\nrel_total is the manuscript's metric; rel_anomaly and RMSE are the ones")
    print("§3.1 argues should lead Table 3. Quote eps_hu in the abstract alongside eps_h.")
    RESULTS["table3"] = table3
    save_results()

In [ ]:
if RUN_PART4:
    ic0 = list(MODELS40)[0]
    fig, ax = plt.subplots(1, 3, figsize=(12, 3.4))
    for a, (cname, (h0a, ba)) in zip(ax, C_SINGLE.items()):
        hr, _ = ref_solution(h0a, ba)
        a.plot(xe, hr, 'k-', lw=1.6, label='WB-HLL reference')
        for ic, mdl_ in MODELS40.items():
            hp, _ = eval_batch(mdl_, h0a[None, :], ba[None, :])
            a.plot(xe, hp[0], '--', lw=1.1, label=f'v6 40k ({ic})')
        a.plot(xe, h0a, ':', color='0.6', lw=1.0, label='$h_0$')
        a.set(xlabel='x [m]', ylabel='h [m]', title=cname)
        a.legend(fontsize=7)
    plt.tight_layout()
    show_fig("p4_cases_at_T")

## 4.3 BC-on/off × IC-mode × residual form

The requested 2×2, crossed with the residual form. Each cell trains the v6 model
with **no data supervision** on the C2 configuration, then asks where it ended up
at $t = 1$ s: at $h_0$ (the "$F=0$ attractor" the manuscript claims) or on the
lake-at-rest state of the same volume (what Part 2 predicts).

`d(h0)` and `d(lake)` are $\|\hat h - \cdot\|_2$ over a 500-point grid;
`-> h0` / `-> lake` reports whichever is closer.

In [ ]:
if RUN_PART4:
    banner(f"4.3  BC x IC x residual factorial, {CFG['PI_FACTORIAL_STEPS']} steps each")

    IDX_C2 = 1                                    # row 1 of the pool is C1-IC + bump bed
    h0b = tf.constant(bundle.H0_s[IDX_C2:IDX_C2 + 1])
    bb_ = tf.constant(bundle.B_s[IDX_C2:IDX_C2 + 1])
    h0g = tf.constant(bundle.H0_grid[IDX_C2:IDX_C2 + 1])
    bg_ = tf.constant(bundle.B_grid[IDX_C2:IDX_C2 + 1])

    xev = v6.x_grid_np
    h0_ev = bundle.H0_grid[IDX_C2]
    b_ev = bundle.B_grid[IDX_C2]
    lake_ev = (h0_ev + b_ev).mean() - b_ev        # same-volume lake at rest

    def run_cell(momentum, ic_mode, use_bc, steps, seed=0):
        tf.keras.utils.set_random_seed(seed)
        mdl_ = v6.PIDeepONetSWE(ic_mode=ic_mode).build_once()
        opt_ = v6.make_optimizer()
        step_ = v6.make_pi_step(mdl_, opt_, momentum=momentum, use_bc=use_bc)
        rng_ = np.random.default_rng(seed)
        Lp = Lb = float("nan")
        for it in range(1, steps + 1):
            tbc = tf.constant(rng_.uniform(0, v6.T_np, v6.N_BC).astype(np.float32))
            xc_ = tf.constant(rng_.uniform(0, v6.L_np, v6.N_COLL).astype(np.float32))
            tc_ = tf.constant(rng_.uniform(0, v6.T_np, v6.N_COLL).astype(np.float32))
            _, Lp_, Lb_, _ = step_(h0b, bb_, h0g, bg_, tbc, xc_, tc_)
            Lp, Lb = float(Lp_), float(Lb_)
            if not np.isfinite(Lp):
                break
        hp, _ = v6.predict_at(mdl_, bundle.H0_s[IDX_C2:IDX_C2 + 1],
                              bundle.B_s[IDX_C2:IDX_C2 + 1], xev, v6.T_np,
                              h0_ev[None, :], b_ev[None, :])
        hp = hp[0]
        return dict(momentum=momentum, ic_mode=ic_mode, bc=use_bc,
                    L_pde=Lp, L_bc=Lb,
                    d_h0=float(np.linalg.norm(hp - h0_ev)),
                    d_lake=float(np.linalg.norm(hp - lake_ev)),
                    finite=bool(np.isfinite(Lp)))

    factorial = []
    print(f"{'residual':<12}{'ic_mode':<9}{'BC':<6}{'L_pde':>12}{'d(h0)':>10}"
          f"{'d(lake)':>10}{'closer':>9}")
    for momentum in ("time_only", "full"):
        for ic_mode in ("paper", "exp"):
            for use_bc in (True, False):
                r = run_cell(momentum, ic_mode, use_bc, CFG["PI_FACTORIAL_STEPS"])
                r["closer"] = "h0" if r["d_h0"] < r["d_lake"] else "lake"
                factorial.append(r)
                print(f"{momentum:<12}{ic_mode:<9}{str(use_bc):<6}{r['L_pde']:>12.3e}"
                      f"{r['d_h0']:>10.4f}{r['d_lake']:>10.4f}{r['closer']:>9}")

    by_res = {m: {r["closer"] for r in factorial if r["momentum"] == m}
              for m in ("time_only", "full")}
    print(f"\ntime_only residual -> {by_res['time_only']}")
    print(f"full residual      -> {by_res['full']}")
    print("If the verdict flips with the residual form but not with BC or ic_mode,")
    print("the h0 attractor is an artefact of the truncated momentum residual.")

    RESULTS["bc_ic_residual_factorial"] = factorial
    save_results()

In [ ]:
if RUN_PART4:
    fig, ax = plt.subplots(1, 2, figsize=(11, 3.6), sharey=True)
    for a, momentum in zip(ax, ("time_only", "full")):
        rows = [r for r in factorial if r["momentum"] == momentum]
        lbl = [f"ic={r['ic_mode']}\nBC={'on' if r['bc'] else 'off'}" for r in rows]
        pos = np.arange(len(rows))
        a.bar(pos - 0.2, [r["d_h0"] for r in rows], 0.4, label=r'$\|\hat h - h_0\|$')
        a.bar(pos + 0.2, [r["d_lake"] for r in rows], 0.4, label=r'$\|\hat h - $lake$\|$')
        a.set_xticks(pos); a.set_xticklabels(lbl, fontsize=7)
        a.set_title(f"residual: {momentum}")
        a.legend(fontsize=8)
    ax[0].set_ylabel("distance at $t=1$ s")
    plt.tight_layout()
    show_fig("p4_factorial_h0_vs_lake")

## 4.4 The PDE gradient norm, measured in v6's own code

§3.7.3 reports $6.6\times10^{12}$, Remark 3 reports $1.5\times10^2$, and the Fig. 6
caption says $2.2\times10^1$. Every row below is measured on the same freshly
initialised v6 model at the same collocation points, varying one protocol choice
at a time, so the spread between the published numbers can be attributed rather
than argued about.

`sum` vs `mean` is included because an unreduced loss is the cheapest explanation
for a norm several orders of magnitude above the others.

In [ ]:
if RUN_PART4:
    banner("4.4  PDE gradient norm under matched protocols (v6 code)")

    rng_g = np.random.default_rng(0)
    xc_g = tf.constant(rng_g.uniform(0, v6.L_np, v6.N_COLL).astype(np.float32))
    tc_g = tf.constant(rng_g.uniform(0, v6.T_np, v6.N_COLL).astype(np.float32))

    def fresh():
        tf.keras.utils.set_random_seed(CFG["SEED"])
        return v6.PIDeepONetSWE(ic_mode="paper").build_once()

    def slice_pool(n):
        s = slice(0, n)
        return (tf.constant(bundle.H0_s[s]), tf.constant(bundle.B_s[s]),
                tf.constant(bundle.H0_grid[s]), tf.constant(bundle.B_grid[s]))

    def measure(label, loss_fn, n_traj):
        mdl_ = fresh()
        a, b_, c_, dd_ = slice_pool(n_traj)
        with tf.GradientTape(persistent=True) as tape:
            Lp = loss_fn(mdl_, a, b_, c_, dd_)
        groups = {"branch_h": mdl_.b1h.trainable_variables + mdl_.b2h.trainable_variables,
                  "branch_hu": mdl_.b1hu.trainable_variables + mdl_.b2hu.trainable_variables,
                  "trunk_h": mdl_.th.trainable_variables,
                  "trunk_hu": mdl_.thu.trainable_variables}
        out = {k: float(tf.linalg.global_norm(
                   [g for g in tape.gradient(Lp, v) if g is not None]))
               for k, v in groups.items()}
        out["total"] = float(tf.linalg.global_norm(
            [g for g in tape.gradient(Lp, mdl_.trainable_variables) if g is not None]))
        out["L_pde"] = float(Lp)
        del tape
        return dict(protocol=label, batch=n_traj, **out)

    fd_time = lambda m, a, b_, c_, d_: v6.pde_residual_fd(
        m, a, b_, c_, d_, xc_g, tc_g, momentum="time_only")
    fd_full = lambda m, a, b_, c_, d_: v6.pde_residual_fd(
        m, a, b_, c_, d_, xc_g, tc_g, momentum="full")
    ad_full = lambda m, a, b_, c_, d_: v6.pde_residual_ad(m, a, b_, c_, d_, xc_g, tc_g)
    fd_time_sum = lambda m, a, b_, c_, d_: v6.pde_residual_fd(
        m, a, b_, c_, d_, xc_g, tc_g, momentum="time_only") * float(
            v6.N_COLL * min(v6.BATCH, bundle.n_pool))

    gnorm_rows = [
        measure("FD, R2 = hu_t (v6 verbatim)", fd_time, v6.BATCH),
        measure("FD, R2 = hu_t",               fd_time, 1),
        measure("FD, full momentum",           fd_full, 1),
        measure("autodiff, full momentum",     ad_full, 1),
        measure("FD, R2 = hu_t, SUM not MEAN", fd_time_sum, v6.BATCH),
    ]

    print(f"{'protocol':<32}{'B':>3}{'L_pde':>12}{'branch_h':>11}{'branch_hu':>11}"
          f"{'trunk_h':>10}{'trunk_hu':>10}{'TOTAL':>12}")
    for r in gnorm_rows:
        print(f"{r['protocol']:<32}{r['batch']:>3}{r['L_pde']:>12.3e}"
              f"{r['branch_h']:>11.3e}{r['branch_hu']:>11.3e}{r['trunk_h']:>10.3e}"
              f"{r['trunk_hu']:>10.3e}{r['total']:>12.3e}")

    tot = {r["protocol"]: r["total"] for r in gnorm_rows}
    print(f"\nspread across protocols: {min(tot.values()):.3e} .. {max(tot.values()):.3e}"
          f"  ({max(tot.values()) / max(min(tot.values()), 1e-30):.1e}x)")
    print("Report one row, name the protocol in the caption, and make §3.7.3,")
    print("Remark 3 and Fig. 6 all quote that same row.")

    RESULTS["pde_gradient_norm_v6"] = gnorm_rows
    save_results()

### Text changes Part 4 supports

- Abstract, Table 1, Table 3 — every headline error number comes from §4.2, on
  well-balanced reference data, with the anomaly-normalised and dimensional
  columns alongside the existing one.
- §3.5.1 / Proposition 1 — state that the $F=0$ attractor is a property of the
  *implemented* residual (§4.3): with $R_2 = \partial_t(hu)$, $F=0$ is an exact
  global minimum; with the full momentum equation it is not, and training leaves
  for the steady-state manifold instead. This is a much stronger and more honest
  result than the current chain-rule argument.
- §3.7.3, Remark 3, Fig. 6 caption — replace all three numbers with one row of the
  §4.4 table and name the protocol.
- Eq. (12) — §4.1 gives the accuracy cost of the corrected IC shortcut, so the
  replacement can be justified rather than asserted.

---

## Run summary

Everything the run produced, with a manifest of the files written to the output
directory. On Kaggle these are downloadable from the **Output** tab of the committed
version.

In [ ]:
banner("Run summary")

RESULTS["wall_clock_seconds"] = time.time() - _T_START
RESULTS["parts_run"] = dict(part1=RUN_PART1, part2=RUN_PART2,
                            part3=RUN_PART3, part4=RUN_PART4)
save_results()

print(f"total wall clock: {RESULTS['wall_clock_seconds'] / 60:.1f} min"
      f"   (QUICK = {QUICK})")
print(f"\noutput directory: {OUT_DIR}")
for p in sorted(OUT_DIR.rglob("*")):
    if p.is_file():
        print(f"  {p.relative_to(OUT_DIR).as_posix():<40} {p.stat().st_size / 1024:9.1f} KB")

print("\nHeadline numbers now in results.json:")
for k in ("cfl_audit", "error_budget", "data_generation", "gradient_split_at_F0",
          "attractor_endpoint", "pde_gradient_norm", "ic_shortcut",
          "operator_conservation", "fusion_ablation", "speedup",
          "run40k", "table3", "bc_ic_residual_factorial", "pde_gradient_norm_v6"):
    print(f"  {'OK ' if k in RESULTS else '-- '} {k}")

if QUICK:
    print("\n*** QUICK mode: these numbers are NOT publication grade. "
          "Set QUICK = False and Save & Run All. ***")